In [6]:
import json
from pathlib import Path
from modules.tools import normalize
import pandas as pd
from xgboost import XGBClassifier

from sklearn.naive_bayes import GaussianNB
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import minmax_scale
from sklearn.model_selection import cross_val_score
from sklearn import tree, svm
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier

from sklearn.metrics import classification_report, roc_curve

In [2]:
def load_config(config_path):
    with open(config_path, "r") as f:
        return json.load(f)

In [3]:
config = load_config(Path("configs/load_data.json"))

load_data_folder = Path(config["preprocess_folder"]) / "tokenized"
save_folder = Path(config["save_models_folder"])
dataset_name = config["dataset_for_experiment"]
features = config["features"]
models = config["models"]
batch_size = config["batch_size"]
epochs_num = config["epochs_num"]
learning_rate = config["learning_rate"]
device = config["device"]
fp16 = config["fp16"]

In [4]:
def divide_data(dataset, features):

    _, preprocessor = normalize(dataset[dataset["split"] == "train"], features=features)
    dataset, preprocessor = normalize(dataset, features=features, preprocessor=preprocessor)

    columns_y = "label"

    X_train = dataset[dataset["split"] == "train"][features]
    X_val = dataset[dataset["split"] == "validate"][features]
    X_test = dataset[dataset["split"] == "test"][features]

    y_train = dataset[dataset["split"] == "train"][columns_y]
    y_val = dataset[dataset["split"] == "validate"][columns_y]
    y_test = dataset[dataset["split"] == "test"][columns_y]

    return X_train, X_val, X_test, y_train, y_val, y_test, preprocessor

In [10]:
gnb = GaussianNB()
lor = LogisticRegression()
knn = KNeighborsClassifier()
raf = RandomForestClassifier()
tr = tree.DecisionTreeClassifier()
cvm = svm.SVC()
xgb = XGBClassifier()
etc = ExtraTreesClassifier()
# models = {
#     "GaussianNB": gnb,
#     "LogisticRegression": lor,
#     "KNeighbors": knn,
#     "RandomForest": raf,
#     "DecisionTree": tr,
#     "CVM": cvm
# }
models = {
    "XGBoost": XGBClassifier(tree_method="hist", n_jobs=1),
    "ExtraTrees": ExtraTreesClassifier(),
}

dataset_path = Path("ai_detection/data/processed/tokenized/bert-base-uncased_MAGE.parquet")
dataset_df = pd.read_parquet(dataset_path, columns=[*features, "split", "label"])

X_train, X_val, X_test, y_train, y_val, y_test, preprocessor = divide_data(dataset_df, features)

for name, model in models.items():
    model.fit(X_train, y_train)

    val_score = model.predict_proba(X_val)[:, 1]
    fpr, recall, thresholds = roc_curve(y_val, val_score, pos_label=1)

    allowed = fpr <= 0.02
    threshold = thresholds[allowed][recall[allowed].argmax()]

    test_score = model.predict_proba(X_test)[:, 1]
    y_pred = (test_score >= threshold).astype(int)

    print(f"{name} | threshold {threshold:.3f}")
    print(classification_report(y_test, y_pred, digits=3))

XGBoost | threshold 0.933
              precision    recall  f1-score   support

           0      0.637     0.978     0.771     30265
           1      0.953     0.446     0.608     30478

    accuracy                          0.711     60743
   macro avg      0.795     0.712     0.689     60743
weighted avg      0.795     0.711     0.689     60743

ExtraTrees | threshold 0.880
              precision    recall  f1-score   support

           0      0.619     0.982     0.759     30265
           1      0.957     0.400     0.564     30478

    accuracy                          0.690     60743
   macro avg      0.788     0.691     0.662     60743
weighted avg      0.789     0.690     0.661     60743

